# 🎥 Real Text-to-Video Test — LTX-Video 2B

This notebook generates a **real video directly from text**. It does not create a still image and animate it.

Test goal: a real human walks naturally, feet/legs and arms move, camera follows physically, then a mystery reveal happens.

**GPU:** Kaggle T4 x2 recommended. This test uses the smaller official LTX-Video 2B distilled checkpoint so it is much more realistic for free Kaggle hardware than the 22B LTX-2.3 checkpoint.

In [ ]:
!nvidia-smi
!git clone -q --depth 1 https://github.com/Lightricks/LTX-Video.git /kaggle/working/LTX-Video
%cd /kaggle/working/LTX-Video
!pip -q install -e .[inference-script] huggingface_hub av imageio[ffmpeg]
!sudo apt-get update -qq && sudo apt-get install -y -qq ffmpeg

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path

MODEL_DIR = Path('/kaggle/working/models')
MODEL_DIR.mkdir(exist_ok=True)
ckpt = hf_hub_download(
    repo_id='Lightricks/LTX-Video',
    filename='ltxv-2b-0.9.6-distilled-04-25.safetensors',
    local_dir=MODEL_DIR,
)
print('Checkpoint:', ckpt)

In [ ]:
import yaml, pathlib

src = pathlib.Path('/kaggle/working/LTX-Video/configs/ltxv-2b-0.9.6-distilled.yaml')
cfg = yaml.safe_load(src.read_text())
cfg['checkpoint_path'] = '/kaggle/working/models/ltxv-2b-0.9.6-distilled-04-25.safetensors'
cfg['precision'] = 'bfloat16'
cfg['num_inference_steps'] = 8
cfg['stochastic_sampling'] = True
test_cfg = pathlib.Path('/kaggle/working/LTX-Video/configs/kaggle-ltx-test.yaml')
test_cfg.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(test_cfg.read_text())

## 🎬 Actual video prompt

This is intentionally written as a continuous cinematography/action description. LTX's documentation recommends literal chronological descriptions of actions, movement and camera behavior.

In [ ]:
PROMPT = '''Photorealistic live-action cinematic video, vertical 9:16. A real young man wearing a dark jacket and jeans walks naturally down an abandoned underground railway platform. His feet visibly take realistic alternating steps, knees bend naturally, arms swing with each step, clothing moves with his walking, and his body weight shifts realistically. The camera begins high and far behind him, then physically descends and smoothly follows him at walking speed, passing close to pillars and foreground objects with strong parallax. The man notices a faint warm light ahead and slows down. He walks toward a dark maintenance doorway. The camera follows directly behind him and then moves around his shoulder as he reaches the doorway. He opens it and suddenly reveals a gigantic hidden underground city filled with distant lights, roads and enormous structures far below. He stops naturally in shock while the camera pushes forward past him toward the massive reveal. Realistic human anatomy, realistic feet and hands, continuous body motion, believable physics, natural cinematic lighting, realistic motion blur, premium live-action photography, no slideshow, no static image, no digital zoom, no CGI-looking person, no text, no logo, no watermark.'''
print(PROMPT)

In [ ]:
import subprocess, shlex, os

out='/kaggle/working/ltx_real_t2v_test.mp4'
cmd=[
  'python','inference.py',
  '--prompt',PROMPT,
  '--height','1024','--width','576',
  '--num_frames','97',
  '--seed','12345',
  '--pipeline_config','configs/kaggle-ltx-test.yaml',
  '--output_path',out,
]
print('Running:', ' '.join(shlex.quote(x) for x in cmd if x != PROMPT))
subprocess.run(cmd, check=True)
print('DONE:', out, os.path.getsize(out))

In [ ]:
from IPython.display import Video, display
display(Video('/kaggle/working/ltx_real_t2v_test.mp4', embed=True, width=360))